# Active Learning â€” Score & Prioritize Unlabelled Audio
Runs the existing ensemble (wav2vec2 + XGBoost) on a new batch of unlabelled audio,
then ranks files by **uncertainty** so you label the most informative ones first.

### Setup
1. Place `audios3/` folder in the same directory as `audios2/`, `audios4/`
2. Make sure models are available (ONNX + XGBoost + scaler)
3. Run cells top to bottom

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
AUDIO_DIR = "audios3"               # new unlabelled batch
TRANSCRIPTS_FILE = "transcripts_audios3.json"
OUTPUT_CSV = "active_learning_audios3.csv"

# Models â€” auto-detect from common locations
def find_file(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

ONNX_MODEL = find_file(
    "wav2vec2_combined_quant.onnx",
    "combined/wav2vec2_combined_quant.onnx",
    "checkpoints_combined/wav2vec2_combined_quant.onnx",
)
XGBOOST_MODEL = find_file(
    "checkpoints_finetuned/xgboost_finetuned.json",
    "xgboost_ensemble.json",
    "ensemble/xgboost_ensemble.json",
    "checkpoints_ensemble/xgboost_ensemble.json",
    "checkpoints_ensemble_textonly/xgboost_ensemble.json",
)
SCALER_PKL = find_file(
    "checkpoints_finetuned/scaler.pkl",
    "scaler.pkl",
    "ensemble/scaler.pkl",
    "checkpoints_ensemble/scaler.pkl",
    "checkpoints_ensemble_textonly/scaler.pkl",
)

# Voting weight (from eval_and_finetune optimization)
WAV2VEC2_WEIGHT = 0.1  # adjust if you found a different optimal

print(f"Audio dir:    {AUDIO_DIR} {'âœ“' if os.path.isdir(AUDIO_DIR) else 'âœ— NOT FOUND'}")
print(f"ONNX model:   {ONNX_MODEL or 'âœ— NOT FOUND'}")
print(f"XGBoost:      {XGBOOST_MODEL or 'âœ— NOT FOUND'}")
print(f"Scaler:       {SCALER_PKL or 'âœ— NOT FOUND'}")

## 1. Scan Audio Files

In [ ]:
# Find all .wav files in audios3/candidate/candidate_*.wav
audio_files = sorted(Path(AUDIO_DIR).rglob("*.wav"))
print(f"Found {len(audio_files)} audio files in {AUDIO_DIR}/")

# Build manifest
manifest = pd.DataFrame({
    "filepath": [str(f) for f in audio_files],
    "filename": [f.name for f in audio_files],
    "candidate": [f.parent.name for f in audio_files],
    "audio_batch": AUDIO_DIR,
})

print(f"Candidates: {manifest['candidate'].nunique()}")
print(f"Files per candidate: {manifest.groupby('candidate').size().describe()[['mean','min','max']].to_dict()}")
manifest.head()

## 2. Transcribe (WhisperX / faster-whisper)
Transcription is required for text + pause features.  
Resumes from saved transcripts if interrupted.

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print(f"Device: {DEVICE}")

# Load existing transcripts if resuming
transcripts = {}
if os.path.exists(TRANSCRIPTS_FILE):
    with open(TRANSCRIPTS_FILE) as f:
        transcripts = json.load(f)
    print(f"Resuming: {len(transcripts)} already transcribed")

to_transcribe = manifest[~manifest["filepath"].isin(transcripts.keys())]
print(f"Files to transcribe: {len(to_transcribe)}")

In [ ]:
if len(to_transcribe) > 0:
    if HAS_GPU:
        import whisperx
        model = whisperx.load_model("large-v2", device=DEVICE, compute_type="float16", language="en")
        model_a, metadata = whisperx.load_align_model(language_code="en", device=DEVICE)

        for _, row in tqdm(to_transcribe.iterrows(), total=len(to_transcribe), desc="Transcribing (GPU)"):
            filepath = row["filepath"]
            try:
                audio = whisperx.load_audio(filepath)
                result = model.transcribe(audio, batch_size=8, language="en")
                try:
                    aligned = whisperx.align(result["segments"], model_a, metadata, audio,
                                             DEVICE, return_char_alignments=False)
                    words = [w for seg in aligned["segments"] for w in seg.get("words", [])]
                except:
                    words = [w for seg in result["segments"] for w in seg.get("words", [])]

                text = " ".join(seg["text"] for seg in result["segments"])
                transcripts[filepath] = {
                    "text": text, "words": words,
                    "filename": row["filename"],
                    "duration_sec": len(audio) / 16000,
                }
            except Exception as e:
                print(f"  Error: {filepath}: {e}")
                transcripts[filepath] = {"text": "", "words": [], "filename": row["filename"], "duration_sec": 0}

            if len(transcripts) % 10 == 0:
                with open(TRANSCRIPTS_FILE, "w") as f:
                    json.dump(transcripts, f, indent=2, ensure_ascii=False)
    else:
        from faster_whisper import WhisperModel
        model = WhisperModel("large-v2", device="cpu", compute_type="int8")

        for _, row in tqdm(to_transcribe.iterrows(), total=len(to_transcribe), desc="Transcribing (CPU)"):
            filepath = row["filepath"]
            try:
                segments, info = model.transcribe(filepath, language="en", word_timestamps=True)
                words, texts = [], []
                for seg in segments:
                    texts.append(seg.text)
                    for w in seg.words or []:
                        words.append({"word": w.word.strip(), "start": w.start, "end": w.end})
                transcripts[filepath] = {
                    "text": " ".join(texts), "words": words,
                    "filename": row["filename"],
                    "duration_sec": info.duration,
                }
            except Exception as e:
                print(f"  Error: {filepath}: {e}")
                transcripts[filepath] = {"text": "", "words": [], "filename": row["filename"], "duration_sec": 0}

            if len(transcripts) % 10 == 0:
                with open(TRANSCRIPTS_FILE, "w") as f:
                    json.dump(transcripts, f, indent=2, ensure_ascii=False)

    with open(TRANSCRIPTS_FILE, "w") as f:
        json.dump(transcripts, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(transcripts)} transcripts -> {TRANSCRIPTS_FILE}")
else:
    print("All files already transcribed.")

## 3. Extract Features (text + pause + prosodic + wav2vec2)

In [ ]:
from extract_features_company import (
    compute_text_features, compute_pause_features, compute_prosodic_features,
    compute_wav2vec2_scores, _empty_pause_features, _empty_prosodic_features,
)

# Load ONNX model for wav2vec2 scoring
ort_session = None
if ONNX_MODEL:
    import onnxruntime as ort
    ort_session = ort.InferenceSession(ONNX_MODEL, providers=["CPUExecutionProvider"])
    print(f"Loaded ONNX: {ONNX_MODEL}")
else:
    print("WARNING: No ONNX model found â€” wav2vec2 scores will be 0")

rows = []
for filepath, t in tqdm(transcripts.items(), desc="Extracting features"):
    text = t.get("text", "")
    words = t.get("words", [])
    filename = t.get("filename", Path(filepath).name)

    text_feats = compute_text_features(text)
    pause_feats = compute_pause_features(words) if words else _empty_pause_features()
    prosodic_feats = compute_prosodic_features(filepath) if os.path.exists(filepath) else _empty_prosodic_features()

    wav2vec2_feats = {"wav2vec2_read_ratio": 0, "wav2vec2_mean_p_read": 0, "wav2vec2_max_p_read": 0}
    if ort_session and os.path.exists(filepath):
        wav2vec2_feats = compute_wav2vec2_scores(filepath, ort_session)

    row = {"filepath": filepath, "filename": filename,
           "candidate": Path(filepath).parent.name,
           "duration_sec": t.get("duration_sec", 0), "text": text[:200]}
    row.update(text_feats)
    row.update(pause_feats)
    row.update(prosodic_feats)
    row.update(wav2vec2_feats)
    rows.append(row)

features_df = pd.DataFrame(rows)
print(f"\nExtracted features for {len(features_df)} files")
print(f"Feature columns: {len([c for c in features_df.columns if c not in ['filepath','filename','candidate','duration_sec','text']])}")

## 4. Score with Both Models

In [ ]:
import xgboost as xgb
import joblib

# Feature columns (text-only for XGBoost, matching training)
TEXT_FEATURES = [
    "filler_rate", "filler_count", "repetition_rate", "repair_rate",
    "ttr", "mattr", "complex_word_rate", "avg_word_length",
    "n_words", "n_unique_words",
    "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
    "self_ref_rate", "discourse_marker_rate", "hedge_rate",
    "noun_rate", "verb_rate", "adj_rate",
]
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
TEXT_ONLY_FEATURES = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES
KNOWN_FEATURES = set(TEXT_ONLY_FEATURES + ["wav2vec2_read_ratio", "wav2vec2_mean_p_read", "wav2vec2_max_p_read"])
META_COLS = {"filepath", "filename", "candidate", "audio_batch", "label", "label_raw",
             "label_int", "duration_sec", "n_transcript_words", "text", "split", "folder"}

# Auto-detect extra numeric features (e.g. mtld)
EXTRA_FEATURES = [c for c in features_df.columns
                  if c not in KNOWN_FEATURES and c not in META_COLS
                  and features_df[c].dtype in ("float64", "float32", "int64", "int32")
                  and features_df[c].fillna(0).std() > 1e-8]
if EXTRA_FEATURES:
    print(f"Auto-detected {len(EXTRA_FEATURES)} extra features: {EXTRA_FEATURES}")

# XGBoost uses text-only + extra features
xgb_features = [c for c in TEXT_ONLY_FEATURES + EXTRA_FEATURES if c in features_df.columns]
print(f"XGBoost features: {len(xgb_features)}")

# Load XGBoost + scaler
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(XGBOOST_MODEL)
scaler = joblib.load(SCALER_PKL)
print(f"Loaded XGBoost: {XGBOOST_MODEL}")
print(f"Loaded scaler:  {SCALER_PKL}")

# XGBoost scores
X = features_df[xgb_features].fillna(0).values
X_scaled = scaler.transform(X)
features_df["xgb_p_cheat"] = xgb_model.predict_proba(X_scaled)[:, 1]

# wav2vec2 scores (already extracted)
features_df["w2v_p_cheat"] = features_df["wav2vec2_mean_p_read"].fillna(0)

# Combined weighted vote
w = WAV2VEC2_WEIGHT
features_df["combined_p_cheat"] = w * features_df["w2v_p_cheat"] + (1 - w) * features_df["xgb_p_cheat"]

print(f"
Scoring summary:")
print(f"  XGBoost  P(cheat): mean={features_df['xgb_p_cheat'].mean():.3f}, std={features_df['xgb_p_cheat'].std():.3f}")
print(f"  wav2vec2 P(cheat): mean={features_df['w2v_p_cheat'].mean():.3f}, std={features_df['w2v_p_cheat'].std():.3f}")
print(f"  Combined P(cheat): mean={features_df['combined_p_cheat'].mean():.3f}, std={features_df['combined_p_cheat'].std():.3f}")

## 5. Uncertainty Ranking
Files closest to the decision boundary (P â‰ˆ 0.5) are **most uncertain** â€” label these first.  
Also samples a few **confident** predictions (high/low P) to catch overconfident mistakes.

In [ ]:
# Uncertainty = distance from 0.5 (lower = more uncertain)
features_df["uncertainty"] = (features_df["combined_p_cheat"] - 0.5).abs()

# Model disagreement = |xgb - w2v| (higher = models disagree)
features_df["model_disagreement"] = (features_df["xgb_p_cheat"] - features_df["w2v_p_cheat"]).abs()

# Priority score: low uncertainty + high disagreement = label first
# Normalize both to [0,1] then combine
features_df["priority"] = (1 - features_df["uncertainty"]) * 0.7 + features_df["model_disagreement"] * 0.3

# Sort by priority (highest = label first)
ranked = features_df.sort_values("priority", ascending=False).reset_index(drop=True)

print("=" * 70)
print("LABELLING PRIORITY (label these first)")
print("=" * 70)
print(f"{'Rank':<5} {'Filename':<35} {'Combined':>8} {'XGB':>6} {'W2V':>6} {'Uncert':>7} {'Disagr':>7}")
print("-" * 70)
for i, row in ranked.head(30).iterrows():
    print(f"{i+1:<5} {row['filename']:<35} {row['combined_p_cheat']:>8.3f} "
          f"{row['xgb_p_cheat']:>6.3f} {row['w2v_p_cheat']:>6.3f} "
          f"{row['uncertainty']:>7.3f} {row['model_disagreement']:>7.3f}")

In [ ]:
# Distribution summary
confident_cheat = (ranked["combined_p_cheat"] >= 0.7).sum()
uncertain = ((ranked["combined_p_cheat"] >= 0.3) & (ranked["combined_p_cheat"] < 0.7)).sum()
confident_clean = (ranked["combined_p_cheat"] < 0.3).sum()

print(f"\nPrediction distribution:")
print(f"  Confident CHEATING  (P >= 0.7): {confident_cheat:>4} ({100*confident_cheat/len(ranked):.0f}%)")
print(f"  UNCERTAIN           (0.3-0.7):  {uncertain:>4} ({100*uncertain/len(ranked):.0f}%)  <-- label ALL of these")
print(f"  Confident CLEAN     (P < 0.3):  {confident_clean:>4} ({100*confident_clean/len(ranked):.0f}%)")

print(f"\nSuggested labelling plan:")
n_uncertain = uncertain
n_confident_sample = min(30, confident_cheat + confident_clean)  # sample ~30 confident ones
print(f"  1. Label ALL {n_uncertain} uncertain files (most informative)")
print(f"  2. Label ~{n_confident_sample} random confident files (catch overconfident errors)")
print(f"  Total: ~{n_uncertain + n_confident_sample} files (out of {len(ranked)})")

## 6. Per-Candidate Summary
Aggregates scores by candidate â€” useful if candidates have multiple audio files.

In [ ]:
candidate_summary = ranked.groupby("candidate").agg(
    n_files=("filename", "count"),
    mean_p_cheat=("combined_p_cheat", "mean"),
    max_p_cheat=("combined_p_cheat", "max"),
    min_p_cheat=("combined_p_cheat", "min"),
    mean_xgb=("xgb_p_cheat", "mean"),
    mean_w2v=("w2v_p_cheat", "mean"),
    mean_uncertainty=("uncertainty", "mean"),
    mean_disagreement=("model_disagreement", "mean"),
).sort_values("mean_uncertainty").reset_index()

print(f"{'Candidate':<25} {'Files':>5} {'Mean P':>7} {'Max P':>7} {'XGB':>6} {'W2V':>6} {'Uncert':>7}")
print("-" * 75)
for _, r in candidate_summary.iterrows():
    flag = " âš " if r["mean_uncertainty"] < 0.2 else ""
    print(f"{r['candidate']:<25} {r['n_files']:>5} {r['mean_p_cheat']:>7.3f} {r['max_p_cheat']:>7.3f} "
          f"{r['mean_xgb']:>6.3f} {r['mean_w2v']:>6.3f} {r['mean_uncertainty']:>7.3f}{flag}")

## 7. Suggested Labelling Batch
Builds the final list: all uncertain files + random sample of confident ones.

In [ ]:
# All uncertain files
uncertain_files = ranked[ranked["uncertainty"] < 0.2].copy()

# Random sample from confident predictions (both cheating and clean)
confident_files = ranked[ranked["uncertainty"] >= 0.2]
n_sample = min(30, len(confident_files))
if n_sample > 0:
    confident_sample = confident_files.sample(n=n_sample, random_state=42)
else:
    confident_sample = pd.DataFrame()

# Combine
to_label = pd.concat([uncertain_files, confident_sample]).drop_duplicates(subset="filepath")
to_label = to_label.sort_values("priority", ascending=False)
to_label["label_reason"] = to_label["uncertainty"].apply(
    lambda u: "uncertain" if u < 0.2 else "confident_audit"
)

print(f"Files to label: {len(to_label)}")
print(f"  Uncertain:      {(to_label['label_reason']=='uncertain').sum()}")
print(f"  Confident audit: {(to_label['label_reason']=='confident_audit').sum()}")

# If you want to label everything, uncomment:
# to_label = ranked.copy(); to_label["label_reason"] = "full_batch"

## 8. Export for Labelling
Saves a CSV you can open in Excel/Sheets to add labels.  
Add your label in the `label` column: `cheating` or `not cheating`.

In [ ]:
# Export labelling sheet
label_cols = ["filepath", "filename", "candidate", "combined_p_cheat",
              "xgb_p_cheat", "w2v_p_cheat", "uncertainty", "model_disagreement",
              "label_reason", "text"]

# Add empty label column for manual annotation
to_label["label"] = ""
to_label["label_int"] = -1  # fill in: 1=cheating, 0=not cheating

export_cols = ["filepath", "filename", "candidate", "label", "label_int",
               "combined_p_cheat", "xgb_p_cheat", "w2v_p_cheat",
               "uncertainty", "label_reason", "text"]
to_label[export_cols].to_csv("labelling_sheet_audios3.csv", index=False)
print(f"Saved: labelling_sheet_audios3.csv ({len(to_label)} files)")
print(f"\nOpen in Excel/Sheets, fill in 'label' column, then re-import for training.")

# Also save full scored dataset
ranked.to_csv(OUTPUT_CSV, index=False)
print(f"Saved full scores: {OUTPUT_CSV} ({len(ranked)} files)")

## 9. After Labelling â€” Merge & Retrain
Once you've labelled the files, run this cell to merge with existing data and retrain.

In [ ]:
# â”€â”€ Load your completed labelling sheet â”€â”€
LABELLED_CSV = "labelling_sheet_audios3.csv"  # edit if you renamed it

new_labels = pd.read_csv(LABELLED_CSV)
new_labels = new_labels[new_labels["label_int"].isin([0, 1])]
print(f"New labels: {len(new_labels)}")
print(f"  Cheating:     {(new_labels['label_int']==1).sum()}")
print(f"  Not cheating: {(new_labels['label_int']==0).sum()}")

# Merge labels back into full features
label_map = dict(zip(new_labels["filepath"], new_labels["label_int"]))
features_df["label_int"] = features_df["filepath"].map(label_map).fillna(-1).astype(int)
new_labelled = features_df[features_df["label_int"].isin([0, 1])].copy()
print(f"\nLabelled audios3 samples with features: {len(new_labelled)}")

# Load existing company features (audios2 + audios4)
EXISTING_FEATURES = find_file("features_company.csv", "features.csv")
if EXISTING_FEATURES:
    existing = pd.read_csv(EXISTING_FEATURES)
    existing = existing[existing["label_int"].isin([0, 1])]
    print(f"Existing labelled data: {len(existing)} ({EXISTING_FEATURES})")

    combined = pd.concat([existing, new_labelled], ignore_index=True)
    combined = combined.drop_duplicates(subset="filepath")
else:
    combined = new_labelled

print(f"\nCombined dataset: {len(combined)}")
print(f"  Cheating:     {(combined['label_int']==1).sum()}")
print(f"  Not cheating: {(combined['label_int']==0).sum()}")

In [ ]:
# ── Retrain XGBoost on combined data ──
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

# Auto-detect extra features in combined data
extra_in_combined = [c for c in combined.columns
                     if c not in KNOWN_FEATURES and c not in META_COLS
                     and combined[c].dtype in ("float64", "float32", "int64", "int32")
                     and combined[c].fillna(0).std() > 1e-8]
avail_features = [c for c in TEXT_ONLY_FEATURES + extra_in_combined
                  if c in combined.columns and combined[c].std() > 1e-8]
print(f"Training features: {len(avail_features)}")
if extra_in_combined:
    print(f"  Including extra: {extra_in_combined}")

X = combined[avail_features].fillna(0).values
y = combined["label_int"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler_new = StandardScaler()
X_train_s = scaler_new.fit_transform(X_train)
X_test_s = scaler_new.transform(X_test)

new_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss", early_stopping_rounds=30,
    random_state=42, use_label_encoder=False,
)
new_model.fit(X_train_s, y_train, eval_set=[(X_test_s, y_test)], verbose=50)

y_pred = new_model.predict(X_test_s)
print(f"
Test F1: {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["not cheating", "cheating"]))

# Cross-validation
cv_scores = cross_val_score(
    xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                       use_label_encoder=False, eval_metric="logloss"),
    scaler_new.fit_transform(X), y, cv=5, scoring="f1"
)
print(f"5-fold CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

In [ ]:
# â”€â”€ Save retrained model â”€â”€
os.makedirs("checkpoints_finetuned", exist_ok=True)
new_model.save_model("checkpoints_finetuned/xgboost_finetuned.json")
joblib.dump(scaler_new, "checkpoints_finetuned/scaler.pkl")

# Save combined features for future use
combined.to_csv("features_company_combined.csv", index=False)

print(f"Saved:")
print(f"  checkpoints_finetuned/xgboost_finetuned.json")
print(f"  checkpoints_finetuned/scaler.pkl")
print(f"  features_company_combined.csv ({len(combined)} rows)")